# Evaluation and offline reports
The primary comparison uses no-hint fully-correct rate on the unchanged, previously explored benchmark. Current-law reviews stay separate. No improvement is claimed before complete compatible paired runs. Core behavior: specs 06, 07 and 09.


In [ ]:
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORPUS = PROJECT / "data/corpora/validated_20261011"
INDEX = PROJECT / "data/indexes/validated-bge_20261011"
BENCHMARK = PROJECT / "data/evaluation"
PROFILES = ("no_rag", "dense", "sparse", "hybrid", "multiquery", "graph", "rerank")
RUN_DIRS = [PROJECT / "data/runs/validated_20261011" / profile for profile in PROFILES]
PILOT_QIDS = ["eval-0002", "eval-0004", "eval-0007", "eval-0053"]
DIAGNOSTIC_DIRS = [PROJECT / "data/diagnostics/validated_20261011/retrieval"]
SAVED_REPORT = PROJECT / "data/reports/validated_20261011"
NEW_RUN = PROJECT / "data/runs/notebook-no-rag"
REPORT = PROJECT / "data/reports/notebook-report"
RUN_EVALUATION = False
WRITE_REPORT = False


## 1. Inspect the frozen benchmark and saved runs
Original answers and references are evaluation inputs only. Missing scores remain missing, never fabricated zeros. The four-intent runs are exploratory; the full benchmark and an independent judge control are absent. Detailed answers, contexts and compatibility gates remain in the saved HTML report.


In [ ]:
from legal_rag.evaluation import load_benchmark
from legal_rag.artifacts import read_json

benchmark = load_benchmark(BENCHMARK)
print({"intents": len(benchmark.questions), "view": benchmark.view, "warnings": benchmark.warnings})
for directory in RUN_DIRS:
    if not (directory / "manifest.json").is_file():
        print({"profile": directory.name, "status": "artifact not available"})
        continue
    manifest = read_json(directory / "manifest.json")
    metrics = read_json(directory / "metrics.json")["no_hint"]
    print({"profile": directory.name, "status": manifest["status"],
           "selected_intents": manifest["selected_intents"], "fully_correct_rate": metrics["fully_correct_rate"],
           "missing_scores": metrics["missing_scores"], "generation_errors": metrics["generation_errors"],
           "known_cost_eur": metrics["usage"]["known_cost_eur"]})
if (SAVED_REPORT / "manifest.json").is_file():
    print({"saved_report": str(SAVED_REPORT / "index.html")})
    print(read_json(SAVED_REPORT / "manifest.json")["warnings"])
else:
    print("The saved report is not available yet at the configured path.")


## 2. Optional four-intent evaluation
Disabled by default. Enabling this calls only gpt-6-luna for generation and judging, with no judge control or retries. The shared ledger caps spending at 0.50 EUR, with 0.10 EUR reserved for validation. Current pricing and credentials are still required. Use a new output directory; this does not overwrite the seven saved runs. Local embeddings are used only if the selected profile needs retrieval.


In [ ]:
if RUN_EVALUATION:
    from legal_rag.cli import main

    exit_code = main(["evaluate", "--profile", "no_rag", "--provider", "openai",
        "--model", "gpt-6-luna", "--judge-provider", "openai", "--judge-model", "gpt-6-luna",
        "--without-control", "--retries", "0", "--budget-eur", "0.5", "--budget-reserve-eur", "0.1",
        "--ledger", str(PROJECT / "data/budget/openai.json"),
        "--corpus", str(CORPUS), "--index", str(INDEX), "--benchmark", str(BENCHMARK),
        "--output", str(NEW_RUN), "--local-files-only", "--qids", *PILOT_QIDS])
    print({"exit_code": exit_code})


## 3. Optional offline report
This writes HTML from saved artifacts and calls no model. Without evaluation runs, the report contains the corpus and saved retrieval diagnostics only. Existing output directories are rejected.


In [ ]:
if WRITE_REPORT:
    from legal_rag.reporting import build_report

    print(build_report(CORPUS, RUN_DIRS, REPORT, diagnostic_dirs=DIAGNOSTIC_DIRS))
